In [ ]:
import pandas as pd 
import numpy as np 
import hashlib
import os
from datetime import datetime

# ==========================================
# Load data
# ==========================================

survey_names = open("survey_listN.txt", "r").read().strip().split('\n')
register = pd.read_excel("public_data_registerN.xlsx")
anon_data = pd.read_csv("anonymised_dataN.csv")

ref_date = datetime(2025, 7, 1)  # Reference date for age calculation (from report)

# ========================================== 
# DOB conversion function
# ==========================================
def dob_to_age_band(dob):
    age = (ref_date - dob).days // 365
    if age < 30:
        return '18-29'
    elif age < 45:
        return '30-44'
    elif age < 60:
        return '45-59'
    else:
        return '60+'

# Function to recode citizenship
def recode_citizenship(cit):
    if cit == 'Denmark':
        return 'Denmark'
    else:
        return 'Other'

# Function to recode marital status
def recode_marital(status):
    if status == 'Widowed' or status == 'Divorced':
        return 'Divorced/Widowed'
    else:
        return status

# Function to convert last_voted to evote
def convert_evote(last_voted):
    if last_voted == 1:
        return 0  # paper
    elif last_voted == 2:
        return 1  # electronic
    else:
        return None  # didn't vote

def find_matches(person_row, anon_df):
    """
    person row: row from register with transformed QIs
    anon_df: anonymised dataset
    Returns: list of matching row indices
    """
    matches = []
    for idx, anon_row in anon_df.iterrows():
        match = True

        # Check each quasi-identifier
        # Sex - direct match
        if person_row['sex'] != anon_row['sex']:
            match = False
            continue
            
        # Age band - direct match
        if person_row['age_band'] != anon_row['age']:
            match = False
            continue

        # ZIP - direct match
        if person_row['zip'] != anon_row['zip']:
            match = False
            continue

        # Citizenship - blank matches anything
        if pd.notna(anon_row['citizenship']) and anon_row['citizenship'] != '':
            #only check if there's an actual value for citizenship
            if person_row['citizenship'] != anon_row['citizenship']:
                match = False
                continue

        # Marital status - blank matches anything
        if pd.notna(anon_row['marital_status']) and anon_row['marital_status'] != '':
            if person_row['marital_status'] != anon_row['marital_status']:
                match = False
                continue    
        
        if match:
            matches.append(idx)
    
    return matches

# ==========================================
# Main attack loop
# ==========================================

identified = []
homogeneity_attacks = []
uncertain = []
no_match = []

for name in survey_names:
    person = register[register['name'] == name]
    if len(person) == 0:
        no_match.append(name)
        continue
    person = person.iloc[0]

    transformed = {
        'sex': person['sex'],
        'age_band': dob_to_age_band(person['dob']),
        'zip': person['zip'],
        'citizenship': recode_citizenship(person['citizenship']),
        'marital_status': recode_marital(person['marital_status'])
    }

    matches = find_matches(transformed, anon_data)

    if len(matches) == 1:
        party = anon_data.loc[matches[0], 'party']
        identified.append({'name': name, 'party': party})
    elif len(matches) > 1:
        parties = anon_data.loc[matches, 'party'].unique()
        if len(parties) == 1:
            homogeneity_attacks.append({'name': name, 'party': parties[0]})
        else:
            uncertain.append({'name': name, 'possible_parties': list(parties)})
    else:
        no_match.append(name)

# ==========================================
# Results Summary
# ==========================================

identified_df = pd.DataFrame(identified)
homogeneity_df = pd.DataFrame(homogeneity_attacks)


print("DE-ANONYMISATION ATTACK RESULTS")

print(f"\nTotal survey respondents: {len(survey_names)}")
print(f"Total votes revealed: {len(identified) + len(homogeneity_attacks)} ({(len(identified) + len(homogeneity_attacks))/2:.1f}%)")

print(f"\n--- Uniquely Identified: {len(identified)} ---")
print(identified_df['party'].value_counts().to_string())

print(f"\n--- Homogeneity Attacks: {len(homogeneity_attacks)} ---")
print(homogeneity_df['party'].value_counts().to_string())

print(f"\n--- Still Protected: {len(uncertain)} ---")
print(f"\n--- No Match in Register: {len(no_match)} ---")

DE-ANONYMISATION ATTACK RESULTS

Total survey respondents: 200
Total votes revealed: 103 (51.5%)

--- Uniquely Identified: 26 ---
party
Green    18
Red       8

--- Homogeneity Attacks: 77 ---
party
Green    66
Red      11

--- Still Protected: 97 ---

--- No Match in Register: 0 ---
